In [0]:
%pip install python-dotenv
dbutils.library.restartPython()

In [0]:
import os
from dotenv import load_dotenv

# 1. Carrega o .env (caminho do Workspace e, como fallback, o padrão)
notebook_path = "/Workspace/Users/nascimento.germano@gmail.com/estagio-empregadados-turma-2"
env_path = os.path.join(notebook_path, ".env")
load_dotenv(dotenv_path=env_path)
load_dotenv()

# 2. Lê do .env; se não achar, tenta Databricks Secrets
SECRET_SCOPE = "squad3"

def obter_config(chave):
    valor = os.getenv(chave)
    if valor:
        return valor
    try:
        return dbutils.secrets.get(scope=SECRET_SCOPE, key=chave)
    except Exception:
        return None

CLIENT_ID       = obter_config("CLIENT_ID")
CLIENT_SECRET   = obter_config("CLIENT_SECRET")
TENANT_ID       = obter_config("TENANT_ID")
STORAGE_ACCOUNT = obter_config("STORAGE_ACCOUNT")
CONTAINER       = obter_config("CONTAINER")

# 3. Validação
if not all([CLIENT_ID, CLIENT_SECRET, TENANT_ID, STORAGE_ACCOUNT, CONTAINER]):
    raise ValueError(
        f"❌ ERRO: configuração incompleta.\n"
        f"Caminho do .env tentado: {env_path} | Secret scope: {SECRET_SCOPE}"
    )

# 4. Endpoint OAuth
OAUTH_ENDPOINT = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token"

# 5. Caminhos ADLS (definidos uma única vez)
RAW_BASE_PATH    = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
BRONZE_BASE_PATH = f"{RAW_BASE_PATH}/bronze"

RAW_ITENS_VENDA_PATH = f"{RAW_BASE_PATH}/batch-data/physical_itens_venda_caixa.csv"
RAW_VENDAS_PATH      = f"{RAW_BASE_PATH}/batch-data/physical_vendas_caixa.csv"

BRONZE_ITENS_VENDA_PATH = f"{BRONZE_BASE_PATH}/squad3/itens_venda"
BRONZE_VENDAS_PATH      = f"{BRONZE_BASE_PATH}/squad3/vendas"

print("✅ Configuração carregada com sucesso!")
print(f"   Storage Account : {STORAGE_ACCOUNT}")
print(f"   Container       : {CONTAINER}")
print(f"   Raw itens venda : {RAW_ITENS_VENDA_PATH}")
print(f"   Raw vendas      : {RAW_VENDAS_PATH}")
print(f"   Bronze itens    : {BRONZE_ITENS_VENDA_PATH}")
print(f"   Bronze vendas   : {BRONZE_VENDAS_PATH}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

# O ambiente Serverless não permite spark.conf.set para fs.azure.*:
# a autenticação OAuth (Service Principal) é passada como opção em cada leitura/escrita.
ADLS_OPTIONS = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": CLIENT_ID,
    "fs.azure.account.oauth2.client.secret": CLIENT_SECRET,
    "fs.azure.account.oauth2.client.endpoint": OAUTH_ENDPOINT,
}

def ler_csv_adls(caminho):
    """Lê CSV do ADLS e registra o arquivo de origem real."""
    return (spark.read
        .format("csv")
        .options(**ADLS_OPTIONS)
        .option("header", "true")
        .option("inferSchema", "true")
        .load(caminho)
        .withColumn("bronze_source_file", F.col("_metadata.file_path")))

def ler_delta(caminho):
    return spark.read.format("delta").options(**ADLS_OPTIONS).load(caminho)

def delta_existe(caminho):
    """True se existe _delta_log no caminho. Outros erros (auth, rede) são propagados."""
    try:
        _ = (spark.read.format("binaryFile").options(**ADLS_OPTIONS)
             .option("pathGlobFilter", "*.json")
             .load(f"{caminho}/_delta_log")
             .columns)  # força a análise do plano (Spark Connect é lazy)
        return True
    except Exception as e:
        marcadores = ["Path does not exist", "PATH_NOT_FOUND", "does not exist", "doesn't exist"]
        if any(m in str(e) for m in marcadores):
            return False
        raise

def arquivo_modificado(raw_path, bronze_path):
    """True se o CSV foi modificado após a última ingestão (ou se a Bronze não existe)."""
    if not delta_existe(bronze_path):
        print("   🆕 Bronze não existe → primeira carga")
        return True

    # Comparação em epoch (segundos), sem problemas de fuso horário
    mod_s = (spark.read.format("binaryFile").options(**ADLS_OPTIONS).load(raw_path)
             .agg(F.max(F.unix_timestamp("modificationTime"))).first()[0])
    ult_s = (ler_delta(bronze_path)
             .agg(F.max(F.unix_timestamp("bronze_ingested_at"))).first()[0])

    print(f"   📄 Arquivo modificado (epoch s): {mod_s} | 🕐 Última ingestão (epoch s): {ult_s}")
    return ult_s is None or mod_s > ult_s

def somente_novos(df, bronze_path, chave):
    """Mantém apenas registros cuja chave ainda não está na Bronze."""
    if delta_existe(bronze_path):
        existentes = ler_delta(bronze_path).select(chave).distinct()
        return df.join(existentes, on=chave, how="left_anti")
    return df

print("✅ Funções criadas.")

In [0]:
df_itens_venda = ler_csv_adls(RAW_ITENS_VENDA_PATH)
df_vendas      = ler_csv_adls(RAW_VENDAS_PATH)

print("✅ Arquivos RAW carregados com sucesso!")
print(f"\nItens de venda: {RAW_ITENS_VENDA_PATH}")
print(f"Vendas:          {RAW_VENDAS_PATH}")

print("\n========== SCHEMA: ITENS VENDA ==========")
df_itens_venda.printSchema()

print("\n========== SCHEMA: VENDAS ==========")
df_vendas.printSchema()

In [0]:
from pyspark.sql.functions import col, count, countDistinct

duplicados_vendas = (
    df_vendas
    .groupBy("id_transacao")
    .agg(count("*").alias("quantidade"),
         countDistinct("dt_venda").alias("datas_distintas"))
    .filter(col("quantidade") > 1)
)

qtd_duplicados = duplicados_vendas.count()

if qtd_duplicados > 0:
    qtd_divergentes = duplicados_vendas.filter(col("datas_distintas") > 1).count()
    print(f"⚠️ {qtd_duplicados} transações duplicadas em df_vendas "
          f"({qtd_divergentes} com datas divergentes).")
    print("   → Será usada a menor dt_venda por transação no join.")
    display(duplicados_vendas.limit(20))
else:
    print("✅ Nenhuma duplicidade encontrada em id_transacao na tabela de vendas.")

In [0]:
df_vendas_data = (
    df_vendas
    .groupBy("id_transacao")
    .agg(F.min("dt_venda").alias("dt_venda"))
)

In [0]:
df_joined = df_itens_venda.join(df_vendas_data, on="id_transacao", how="left")

In [0]:
print("⚙️ Aplicando metadados de auditoria e colunas de partição...")

def adicionar_auditoria_e_particao(df, coluna_data="dt_venda"):
    return (df
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("ano_venda", F.coalesce(F.year(F.col(coluna_data)),  F.lit(0)))
        .withColumn("mes_venda", F.coalesce(F.month(F.col(coluna_data)), F.lit(0))))

# Itens: dt_venda só serve para particionar, então é removida depois
df_itens_bronze  = adicionar_auditoria_e_particao(df_joined).drop("dt_venda")

# Vendas: dado cru, sem deduplicar, mantendo dt_venda
df_vendas_bronze = adicionar_auditoria_e_particao(df_vendas)

print("✨ Colunas calculadas com sucesso!")
df_itens_bronze.printSchema()
df_vendas_bronze.printSchema()

In [0]:
print("💾 Gravando dados na camada Bronze (incremental)...")

# Usa as versões OAuth-aware (delta_existe / ler_delta) da Cell 3;
destinos = [
    (df_itens_bronze,  RAW_ITENS_VENDA_PATH, BRONZE_ITENS_VENDA_PATH, "itens_venda", "id_item_venda"),
    (df_vendas_bronze, RAW_VENDAS_PATH,      BRONZE_VENDAS_PATH,      "vendas",      "id_transacao"),
]

for df, raw, bronze, nome, chave in destinos:
    print(f"\n📥 {nome}")
    if not arquivo_modificado(raw, bronze):
        print("   ℹ️ Arquivo não modificado desde a última ingestão — nada a gravar.")
        continue

    (somente_novos(df, bronze, chave)
        .write
        .format("delta")
        .mode("append")
        .options(**ADLS_OPTIONS)
        .partitionBy("ano_venda", "mes_venda")
        .save(bronze))
    print(f"   ✅ {nome} gravada em {bronze}")

In [0]:
for nome, caminho in [("itens_venda", BRONZE_ITENS_VENDA_PATH), ("vendas", BRONZE_VENDAS_PATH)]:
    df_check = ler_delta(caminho)

    total    = df_check.count()
    sem_data = df_check.filter((F.col("ano_venda") == 0) & (F.col("mes_venda") == 0)).count()
    pct      = (sem_data / total * 100) if total else 0

    print(f"\n===== {nome} =====")
    print(f"Total de registros                    : {total}")
    print(f"Registros sem dt_venda (partição 0/0) : {sem_data} ({pct:.2f}%)")
    if pct > 5:  # limite ajustável
        print("⚠️ Percentual alto na partição 0/0: verifique o formato de dt_venda e o join com vendas.")

    display(df_check.groupBy("ano_venda", "mes_venda").count().orderBy("ano_venda", "mes_venda"))